# 01 - Download BCB SGS data

Download all manually verified Banco Central do Brasil SGS series listed in `data/series_dictionary.csv`.

In [ ]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bcb_api import fetch_sgs_series, save_series_csv

In [ ]:
START_DATE = "2011-01-01"
END_DATE = date.today().isoformat()

SERIES_DICTIONARY_PATH = PROJECT_ROOT / "data" / "series_dictionary.csv"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
COMBINED_OUTPUT_PATH = RAW_DATA_DIR / "bcb_sgs_all_long.csv"

In [ ]:
series_dictionary = pd.read_csv(SERIES_DICTIONARY_PATH)
verified_series = series_dictionary.loc[
    series_dictionary["verified"].astype(str).str.lower().eq("yes")
].copy()

verified_series[["name", "series_id", "frequency", "unit"]]

In [ ]:
downloaded = []
summary_rows = []

for row in verified_series.itertuples(index=False):
    df = fetch_sgs_series(
        series_id=row.series_id,
        start_date=START_DATE,
        end_date=END_DATE,
        name=row.name,
    )

    output_path = RAW_DATA_DIR / f"{row.name}.csv"
    save_series_csv(df, output_path)
    downloaded.append(df)

    summary_rows.append(
        {
            "series": row.name,
            "series_id": row.series_id,
            "observations": len(df),
            "first_date": df["date"].min() if not df.empty else pd.NaT,
            "last_date": df["date"].max() if not df.empty else pd.NaT,
        }
    )

if downloaded:
    combined = pd.concat(downloaded, ignore_index=True)
else:
    combined = pd.DataFrame(columns=["date", "value", "series"])

save_series_csv(combined, COMBINED_OUTPUT_PATH)

summary = pd.DataFrame(summary_rows)
summary["first_date"] = pd.to_datetime(summary["first_date"]).dt.date
summary["last_date"] = pd.to_datetime(summary["last_date"]).dt.date

print(summary.to_string(index=False))